In [32]:
import pandas as pd 
import numpy as np 
import os
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

os.chdir("/home/onyxia/work/PESSD") 

In [34]:
# Création fichier "Part des +65ans"


pop_65 = "data/Pop+65ans.xlsx"
pop_tot = "data/Pop_tot.xlsx"

# Charger les fichiers Excel
pop_65 = pd.read_excel(pop_65, index_col=0)
pop_tot = pd.read_excel(pop_tot, index_col=0)

# Remplacer ":" par NaN
pop_65.replace(":", np.nan, inplace=True)
pop_tot.replace(":", np.nan, inplace=True)

# Convertir en float pour calcul
pop_65 = pop_65.astype(float)
pop_tot = pop_tot.astype(float)

# Calculer la part des +65 ans
part_65 = pop_65 / pop_tot

# Sauvegarder le résultat en Excel dans le même dossier
part_65.to_excel("data/part_pop_plus_65.xlsx", index=True)



/tmp/ipykernel_2436/781214399.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pop_65.replace(":", np.nan, inplace=True)
/tmp/ipykernel_2436/781214399.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pop_tot.replace(":", np.nan, inplace=True)


In [36]:
# Création fichier "Dépenses de santé en % du PIB"


Depenses_sante_vol = "data/Depenses_sante_en_volume.xlsx"
PIB = "data/PIB.xlsx"

# Charger les fichiers Excel
Dep_sante_vol = pd.read_excel(Depenses_sante_vol, index_col=0)
PIB = pd.read_excel(PIB, index_col=0)

# Remplacer ":" par NaN
Dep_sante_vol.replace(":", np.nan, inplace=True)
PIB.replace(":", np.nan, inplace=True)

# Convertir en float pour calcul 
Dep_sante_vol = Dep_sante_vol.astype(float)  
PIB = PIB.astype(float)


# Calculer les dépenses en points de PIB
Dep_PIB = Dep_sante_vol / PIB

# Sauvegarder le résultat en Excel dans le même dossier
Dep_PIB.to_excel("data/Depenses_Sante_PIB.xlsx", index=True)

/tmp/ipykernel_2436/3680042912.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Dep_sante_vol.replace(":", np.nan, inplace=True)
/tmp/ipykernel_2436/3680042912.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  PIB.replace(":", np.nan, inplace=True)


In [37]:
#Création fichier Population par tranche d'âge pour TTD 

INPUT_PATH  = "data/Population_par_age.xlsx"
OUTPUT_PATH = "data/Population_par_age_tranches5ans.xlsx"

wb_in = openpyxl.load_workbook(INPUT_PATH)
ws_in = wb_in.active

# ── Structure du fichier source ──
row1 = [cell.value for cell in ws_in[1]]
year_cols = {}
for i, v in enumerate(row1):
    if v is not None and v != 'TIME':
        year_cols[str(v)] = i + 1  # 1-indexé

countries = []
for row in ws_in.iter_rows(min_row=3, max_row=23, min_col=1, max_col=1, values_only=True):
    if row[0] is not None:
        countries.append(row[0])
country_rows = {c: 3 + i for i, c in enumerate(countries)}

bands = [
    ('65-69', list(range(65, 70))),
    ('70-74', list(range(70, 75))),
    ('75-79', list(range(75, 80))),
    ('80-84', list(range(80, 85))),
    ('85-89', list(range(85, 90))),
    ('90-94', list(range(90, 95))),
    ('95-99', list(range(95, 100))),
]
band_labels = [b[0] for b in bands]
years = sorted(year_cols.keys())

# ── Agrégation par tranche de 5 ans ──
# Valeurs manquantes ":" → 0 dans la tranche
data = {}
for year in years:
    start_col = year_cols[year]
    data[year] = {}
    for country in countries:
        row_idx = country_rows[country]
        data[year][country] = {}
        for band_label, ages in bands:
            total = 0
            has_missing = False
            for age in ages:
                col_idx = start_col + (age - 65)
                val = ws_in.cell(row=row_idx, column=col_idx).value
                if val == ':' or val is None:
                    has_missing = True
                else:
                    try:
                        total += int(val)
                    except (ValueError, TypeError):
                        has_missing = True
            data[year][country][band_label] = 0 if has_missing else total

# ── Création du fichier de sortie ──
wb_out = openpyxl.Workbook()
ws_out = wb_out.active
ws_out.title = "Population_tranches_5ans"

# Styles
hdr_year_fill = PatternFill("solid", start_color="1F4E79", end_color="1F4E79")
hdr_band_fill = PatternFill("solid", start_color="2E75B6", end_color="2E75B6")
country_fill  = PatternFill("solid", start_color="D6E4F0", end_color="D6E4F0")
white_fill    = PatternFill("solid", start_color="FFFFFF", end_color="FFFFFF")
hdr_font_w    = Font(name="Arial", bold=True, color="FFFFFF", size=10)
hdr_font_b    = Font(name="Arial", bold=True, color="FFFFFF", size=9)
country_font  = Font(name="Arial", bold=True, size=9)
data_font     = Font(name="Arial", size=9)
center        = Alignment(horizontal="center", vertical="center")
left          = Alignment(horizontal="left", vertical="center")
thin          = Side(style="thin", color="AAAAAA")
border        = Border(left=thin, right=thin, top=thin, bottom=thin)

# Ligne 1 : en-têtes années (fusionnées)
ws_out.cell(row=1, column=1, value="Pays / Année").font = hdr_font_w
ws_out.cell(row=1, column=1).fill      = hdr_year_fill
ws_out.cell(row=1, column=1).alignment = center
ws_out.cell(row=1, column=1).border    = border

n_bands = len(band_labels)
for yi, year in enumerate(years):
    col_start = 2 + yi * n_bands
    col_end   = col_start + n_bands - 1
    ws_out.merge_cells(start_row=1, start_column=col_start, end_row=1, end_column=col_end)
    cell = ws_out.cell(row=1, column=col_start, value=year)
    cell.font = hdr_font_w; cell.fill = hdr_year_fill
    cell.alignment = center; cell.border = border

# Ligne 2 : en-têtes tranches
cell = ws_out.cell(row=2, column=1, value="Pays")
cell.font = Font(name="Arial", bold=True, color="FFFFFF", size=9)
cell.fill = hdr_band_fill; cell.alignment = center; cell.border = border

for yi in range(len(years)):
    for bi, band in enumerate(band_labels):
        col  = 2 + yi * n_bands + bi
        cell = ws_out.cell(row=2, column=col, value=band)
        cell.font = hdr_font_b; cell.fill = hdr_band_fill
        cell.alignment = center; cell.border = border

# Lignes 3+ : données par pays
for ci, country in enumerate(countries):
    row  = 3 + ci
    fill = country_fill if ci % 2 == 0 else white_fill
    cell = ws_out.cell(row=row, column=1, value=country)
    cell.font = country_font; cell.fill = fill
    cell.alignment = left; cell.border = border
    for yi, year in enumerate(years):
        for bi, band in enumerate(band_labels):
            col  = 2 + yi * n_bands + bi
            cell = ws_out.cell(row=row, column=col, value=data[year][country][band])
            cell.font = data_font; cell.fill = fill
            cell.alignment = center; cell.border = border
            cell.number_format = '#,##0'

# Largeurs de colonnes
ws_out.column_dimensions['A'].width = 18
for yi in range(len(years)):
    for bi in range(n_bands):
        ws_out.column_dimensions[get_column_letter(2 + yi * n_bands + bi)].width = 11

ws_out.freeze_panes = "B3"
ws_out.row_dimensions[1].height = 22
ws_out.row_dimensions[2].height = 18

wb_out.save(OUTPUT_PATH)
print(f"Fichier sauvegardé : {OUTPUT_PATH}")
print(f"Dimensions : {ws_out.max_row} lignes x {ws_out.max_column} colonnes")
print(f"({len(countries)} pays x {len(years)} années x {n_bands} tranches)")


Fichier sauvegardé : data/Population_par_age_tranches5ans.xlsx
Dimensions : 23 lignes x 218 colonnes
(21 pays x 31 années x 7 tranches)
